In [1]:
import numpy as np
import pandas as pd

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score 
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import KBinsDiscretizer 
from sklearn.compose import ColumnTransformer

In [4]:
df=pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])

In [9]:
df.dropna(inplace=True)

In [10]:
df.shape

(714, 3)

In [11]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [13]:
X_train,X_test,Y_train,Y_test=train_test_split(df.drop(columns=['Survived']),
                                               df['Survived'],
                                               test_size=0.2,
                                               random_state=42
                                              )

In [14]:
X_train.head()

,Age,Fare
328,31.0,20.5250
73,26.0,14.4542
253,30.0,16.1000
719,33.0,7.7750
666,25.0,13.0000


In [18]:
clf=DecisionTreeClassifier()

In [19]:
clf.fit(X_train,Y_train)
y_pred=clf.predict(X_test)


In [20]:
accuracy_score(Y_test,y_pred)

0.6223776223776224

In [21]:
Kbin_age=KBinsDiscretizer(n_bins=10,encode='ordinal',strategy='quantile')
Kbin_fare=KBinsDiscretizer(n_bins=10,encode='ordinal',strategy='quantile')

In [28]:
trf=ColumnTransformer([
                   ('first',Kbin_age,[0]),
                   ('second',Kbin_fare,[1])
])

In [30]:
X_train_trf=trf.fit_transform(X_train)
X_test_trf=trf.transform(X_test)


C:\Users\Asif Computer\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\Asif Computer\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [41]:
trf.named_transformers_['first'].bin_edges_

array([array([ 0.42, 14.  , 19.  , 22.  , 25.  , 28.5 , 32.  , 36.  , 42.  ,
              50.  , 80.  ])                                                ],
      dtype=object)

In [44]:
output=pd.DataFrame({
    'age':X_train['Age'],
    'age_trf':X_train_trf[:,0],
    'fare':X_train['Fare'],
    'fare_trf':X_train_trf[:,1]
})

In [45]:
output['age_labels']=pd.cut(x=X_train['Age'],
                                    bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_labels']=pd.cut(x=X_train['Fare'],
                                    bins=trf.named_transformers_['second'].bin_edges_[0].tolist())


In [46]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels,fare_labels
221,27.0,4.0,13.00,4.0,"(25.0, 28.5]","(9.225, 13.0]"
290,26.0,4.0,78.85,8.0,"(25.0, 28.5]","(51.479, 82.171]"
539,22.0,3.0,49.50,7.0,"(19.0, 22.0]","(29.125, 51.479]"
434,50.0,9.0,55.90,8.0,"(42.0, 50.0]","(51.479, 82.171]"
441,20.0,2.0,9.50,3.0,"(19.0, 22.0]","(9.225, 13.0]"


In [47]:
clf=DecisionTreeClassifier()
clf.fit(X_train_trf,Y_train)
y_pred2=clf.predict(X_test_trf)

In [49]:
accuracy_score(Y_test,y_pred2)

0.6223776223776224